# Trial Run 3 Scale Monitor

Launch and monitor sharded hybrid YOLO + SAM3 + CellSeg1 inference across many tiles.

In [ ]:
from pathlib import Path
import os, sys, time, subprocess
import pandas as pd
from IPython.display import Image as IPyImage, FileLink, Markdown, clear_output, display

# ---- User configuration ----
REPO = Path.home() / "Desktop/inference_yolo_sam_cellseg"
SCRIPT = REPO / "trial_run_3_hybrid_inference.py"

TILE_DIR = Path.home() / "Desktop/prototype5_whole_image_runs/target_region_6262_13752um/tiles"
BASE_OUT = Path.home() / "Desktop/prototype5_whole_image_runs/target_region_6262_13752um/trial_run_3_scale"

# One GPU: start with N_WORKERS=1. Try 2 only if memory allows.
# Multi-GPU example: GPU_IDS = [0, 1, 2, 3]
N_WORKERS = 1
GPU_IDS = [0]

# External assets.
SAM3_REPO = Path.home() / "Desktop/sam31-cgh-training-data/sam3"
SAM31_OUTPUT_ROOT = Path.home() / "Desktop/sam31-cgh-strategy2/outputs/strategy2_41tiles_full_unfreeze_20260625_160623"
SAM31_CHECKPOINT = SAM31_OUTPUT_ROOT / "checkpoints/checkpoint.pt"
YOLO_MODEL_PATH = Path.home() / "Desktop/sam31-cgh-training-data/training_data/reference_models/cellseg1_cgh_p2_yolo_best.pt"
CELLSEG1_REPO = Path.home() / "Desktop/1.Data/training_pa_he_annotation_full/outputs/cellseg1_cluster_live/cellseg1_repo"
CELLSEG1_RUN_DIR = Path.home() / "Desktop/1.Data/training_pa_he_annotation_full/outputs/cellseg1_cluster_live/cellseg1_cgh_p2_41full_20260625_124306"

BASE_OUT.mkdir(parents=True, exist_ok=True)
assert SCRIPT.exists(), SCRIPT
assert TILE_DIR.exists(), TILE_DIR

print("REPO:", REPO)
print("SCRIPT:", SCRIPT)
print("TILE_DIR:", TILE_DIR)
print("BASE_OUT:", BASE_OUT)
print("N_WORKERS:", N_WORKERS)
print("GPU_IDS:", GPU_IDS)


In [ ]:
# ---- Helpers used by launch, monitor, log tail, and merge cells ----
IMAGE_SUFFIXES = {".png", ".jpg", ".jpeg", ".tif", ".tiff"}

def list_tiles():
    return sorted(p for p in TILE_DIR.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES and not p.name.startswith("._"))

ALL_TILES = list_tiles()
TOTAL_TILES = len(ALL_TILES)

def worker_out_dir(worker_id):
    return BASE_OUT / f"worker_{worker_id:02d}"

def worker_log_path(worker_id):
    return BASE_OUT / f"worker_{worker_id:02d}.log"

def assigned_count(worker_id, n_workers=N_WORKERS):
    return sum(1 for idx, _ in enumerate(ALL_TILES) if idx % n_workers == worker_id)

def count_completed(worker_id):
    pred_dir = worker_out_dir(worker_id) / "pred_masks"
    return len(list(pred_dir.glob("*_hybrid_instance_mask.png"))) if pred_dir.exists() else 0

def safe_read_csv(path):
    if not path.exists() or path.stat().st_size == 0:
        return pd.DataFrame()
    try:
        df = pd.read_csv(path)
    except Exception:
        return pd.DataFrame()
    if list(df.columns) == ["empty"]:
        return pd.DataFrame()
    return df

def worker_csv(worker_id, name):
    return worker_out_dir(worker_id) / name

def read_all_worker_csv(name):
    frames = []
    for worker_id in range(N_WORKERS):
        df = safe_read_csv(worker_csv(worker_id, name))
        if not df.empty:
            df["worker_id"] = worker_id
            frames.append(df)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

def proc_for_worker(worker_id):
    return PROCS.get(worker_id) if "PROCS" in globals() else None

print("Total tiles found:", TOTAL_TILES)
print("Assigned per worker:", {i: assigned_count(i) for i in range(N_WORKERS)})


In [ ]:
# ---- Launch sharded workers ----
# Re-running this cell will not relaunch workers that are still running in this kernel.
PROCS = globals().get("PROCS", {})
RUN_STARTED_AT = globals().get("RUN_STARTED_AT", time.time())

for worker_id in range(N_WORKERS):
    existing = PROCS.get(worker_id)
    if existing is not None and existing.poll() is None:
        print(f"worker {worker_id:02d} already running pid={existing.pid}")
        continue

    out_dir = worker_out_dir(worker_id)
    out_dir.mkdir(parents=True, exist_ok=True)
    log_path = worker_log_path(worker_id)

    gpu_id = GPU_IDS[worker_id % len(GPU_IDS)]
    env = os.environ.copy()
    env.update({
        "CUDA_VISIBLE_DEVICES": str(gpu_id),
        "PYTHONUNBUFFERED": "1",
        "TRIAL3_IMAGE_DIR": str(TILE_DIR),
        "TRIAL3_TILE_KEYS": "ALL",
        "TRIAL3_SHARD_ID": str(worker_id),
        "TRIAL3_NUM_SHARDS": str(N_WORKERS),
        "TRIAL3_SKIP_EXISTING": "1",
        "TRIAL3_OUT_DIR": str(out_dir),
        "SAM3_REPO": str(SAM3_REPO),
        "SAM31_OUTPUT_ROOT": str(SAM31_OUTPUT_ROOT),
        "SAM31_CHECKPOINT": str(SAM31_CHECKPOINT),
        "YOLO_MODEL_PATH": str(YOLO_MODEL_PATH),
        "CELLSEG1_REPO": str(CELLSEG1_REPO),
        "CELLSEG1_RUN_DIR": str(CELLSEG1_RUN_DIR),
    })

    log_handle = log_path.open("a", buffering=1)
    cmd = [sys.executable, str(SCRIPT)]
    proc = subprocess.Popen(cmd, cwd=str(REPO), env=env, stdout=log_handle, stderr=subprocess.STDOUT, text=True)
    PROCS[worker_id] = proc
    print(f"launched worker {worker_id:02d} gpu={gpu_id} pid={proc.pid} out={out_dir} log={log_path}")

RUN_STARTED_AT = RUN_STARTED_AT
print("Launch complete.")


In [ ]:
# ---- Live dashboard. Stop this cell manually when you are done watching. ----
REFRESH_SECONDS = 5

def format_seconds(seconds):
    if seconds is None or pd.isna(seconds) or seconds == float("inf"):
        return "n/a"
    seconds = max(0, int(seconds))
    h, rem = divmod(seconds, 3600)
    m, s = divmod(rem, 60)
    return f"{h:02d}:{m:02d}:{s:02d}"

def dashboard_once():
    clear_output(wait=True)
    now = time.time()
    elapsed = now - RUN_STARTED_AT

    status_rows = []
    total_assigned = 0
    total_completed = 0
    for worker_id in range(N_WORKERS):
        proc = proc_for_worker(worker_id)
        running = proc is not None and proc.poll() is None
        return_code = None if proc is None else proc.poll()
        assigned = assigned_count(worker_id)
        completed = count_completed(worker_id)
        total_assigned += assigned
        total_completed += completed
        worker_rate = completed / elapsed if elapsed > 0 else 0.0
        remaining = max(assigned - completed, 0)
        eta = remaining / worker_rate if worker_rate > 0 else None
        metrics_rows = len(safe_read_csv(worker_csv(worker_id, "trial_run_3_metrics.csv")))
        morph_rows = len(safe_read_csv(worker_csv(worker_id, "trial_run_3_morphology_features.csv")))
        status_rows.append({
            "worker_id": worker_id,
            "gpu_id": GPU_IDS[worker_id % len(GPU_IDS)],
            "pid": None if proc is None else proc.pid,
            "status": "running" if running else "finished" if proc is not None else "not launched",
            "return_code": return_code,
            "images_assigned": assigned,
            "images_completed": completed,
            "ETA": format_seconds(eta),
            "elapsed": format_seconds(elapsed),
            "metrics_rows": metrics_rows,
            "morphology_rows": morph_rows,
            "log_path": str(worker_log_path(worker_id)),
            "output_dir": str(worker_out_dir(worker_id)),
        })

    tiles_per_sec = total_completed / elapsed if elapsed > 0 else 0.0
    remaining_total = max(total_assigned - total_completed, 0)
    eta_total = remaining_total / tiles_per_sec if tiles_per_sec > 0 else None

    metrics_df = read_all_worker_csv("trial_run_3_metrics.csv")
    morph_df = read_all_worker_csv("trial_run_3_morphology_features.csv")
    cells_per_sec = len(morph_df) / elapsed if elapsed > 0 else 0.0

    display(Markdown(f"""
## Trial Run 3 Live Dashboard

**Throughput:** `{tiles_per_sec:.3f}` tiles/sec | `{cells_per_sec:.3f}` cells/sec  
**Progress:** `{total_completed}/{total_assigned}` tiles | **Estimated remaining:** `{format_seconds(eta_total)}`  
**Elapsed:** `{format_seconds(elapsed)}` | **Refresh:** `{REFRESH_SECONDS}s`
"""))

    display(Markdown("### Section 1 — Live worker status"))
    display(pd.DataFrame(status_rows))

    display(Markdown("### Section 2 — Latest inference images"))
    image_paths = []
    for worker_id in range(N_WORKERS):
        image_paths.extend((worker_out_dir(worker_id) / "comparison_images").glob("*_trial3_compare.png"))
    image_paths = sorted(image_paths, key=lambda p: p.stat().st_mtime if p.exists() else 0, reverse=True)[:5]
    if not image_paths:
        display(Markdown("No comparison images completed yet."))
    for path in image_paths:
        display(FileLink(str(path), result_html_prefix="Open: "))
        display(IPyImage(filename=str(path), width=1100))

    display(Markdown("### Section 3 — Live quantitative metrics"))
    if metrics_df.empty:
        display(Markdown("No metrics CSV rows yet."))
    else:
        metric_cols = ["dice", "iou", "precision", "recall_sensitivity", "specificity"]
        summary = metrics_df.groupby("model", dropna=False)[metric_cols].mean(numeric_only=True).reset_index()
        summary = summary.rename(columns={"recall_sensitivity": "recall"})
        display(summary)

    display(Markdown("### Section 4 — Morphology progress"))
    if morph_df.empty:
        display(Markdown("No morphology CSV rows yet."))
    else:
        total_cells = len(morph_df)
        clear_cells = int((morph_df.get("final_class_name") == "clear_cell_boundary").sum()) if "final_class_name" in morph_df else 0
        compact_cells = int((morph_df.get("final_class_name") == "compact_cell_boundary").sum()) if "final_class_name" in morph_df else 0
        morph_summary = pd.DataFrame([{
            "total_cells_processed": total_cells,
            "clear_cells": clear_cells,
            "compact_cells": compact_cells,
            "average_nc_ratio": pd.to_numeric(morph_df.get("nc_ratio"), errors="coerce").mean(),
            "average_cell_area_px": pd.to_numeric(morph_df.get("cell_area_px"), errors="coerce").mean(),
        }])
        display(morph_summary)

while True:
    dashboard_once()
    time.sleep(REFRESH_SECONDS)


In [ ]:
# ---- Print last 5000 characters of any worker log ----
WORKER_ID = 0
LOG_CHARS = 5000
log_path = worker_log_path(WORKER_ID)
if not log_path.exists():
    print("Log does not exist yet:", log_path)
else:
    text = log_path.read_text(errors="replace")
    print(text[-LOG_CHARS:])


In [ ]:
# ---- Merge worker CSV outputs. Safe to run while workers are still running. ----
MERGE_SPECS = {
    "trial_run_3_metrics.csv": "merged_trial_run_3_metrics.csv",
    "trial_run_3_source_instances.csv": "merged_trial_run_3_source_instances.csv",
    "trial_run_3_final_instances.csv": "merged_trial_run_3_final_instances.csv",
    "trial_run_3_morphology_features.csv": "merged_trial_run_3_morphology_features.csv",
}

for src_name, out_name in MERGE_SPECS.items():
    merged = read_all_worker_csv(src_name)
    out_path = BASE_OUT / out_name
    if merged.empty:
        print(f"No rows yet for {src_name}; skipped {out_path}")
        continue
    merged.to_csv(out_path, index=False)
    print(f"Wrote {len(merged):,} rows -> {out_path}")
